# Chapitre 11 - Cours 2 : ajuster un réseau de neurones (rétropropagation, weight decay)

Ce notebook couvre les sections 11.4 et 11.5 du livre :

1. les critères d'ajustement (erreur quadratique, entropie croisée, éq. (11.9)–(11.10)) ;
2. la **rétropropagation** du gradient, dérivée pas à pas (éq. (11.11)–(11.15)), implémentée en
   numpy puis **vérifiée** par différences finies ;
3. la descente de gradient (éq. (11.13)) : par lots, en ligne, notion d'époque ;
4. les questions pratiques de la section 11.5 : valeurs initiales, sur-ajustement et arrêt
   précoce, **weight decay** (éq. (11.16)–(11.17)), mise à l'échelle des entrées, minima
   multiples ;
5. l'exemple du **mélange** du chapitre 2 (figures 11.4 et 11.5).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl.ch11_reseaux import (MLP, backprop, forward, gradient_check, init_params,
                              numerical_gradient, one_hot, pack, sigmoid, unpack)

## 1. Le critère à minimiser

Rappelons le modèle (11.5) : pour une observation $x_i$,

$$z_{mi} = \sigma(\alpha_{0m} + \alpha_m^T x_i), \qquad T_{ki} = \beta_{0k} + \beta_k^T z_i, \qquad f_k(x_i) = g_k(T_i),$$

où $z_i = (z_{1i}, \dots, z_{Mi})$. L'ensemble $\theta$ des poids (on dit souvent *poids* pour
*paramètres*) comporte $M(p+1) + K(M+1)$ nombres (éq. (11.8)).

* En **régression**, on minimise la somme des carrés des erreurs

$$R(\theta) = \sum_{k=1}^K \sum_{i=1}^N \big(y_{ik} - f_k(x_i)\big)^2 \qquad \text{(éq. (11.9))}.$$

* En **classification**, avec $y_{ik} = 1$ si l'observation $i$ est de classe $k$ et 0 sinon, on
  utilise soit l'erreur quadratique, soit l'**entropie croisée** (ou déviance)

$$R(\theta) = -\sum_{i=1}^N \sum_{k=1}^K y_{ik} \log f_k(x_i) \qquad \text{(éq. (11.10))},$$

  et le classifieur est $G(x) = \arg\max_k f_k(x)$.

> **Remarque.** Avec la sortie softmax et l'entropie croisée, $R(\theta)$ est exactement
> l'opposé de la log-vraisemblance d'un modèle logistique multinomial **en les unités
> cachées** $z_m$ : ajuster le réseau, c'est faire du maximum de vraisemblance.
>
> En général on ne veut **pas** le minimum global de $R$, qui sur-ajuste presque toujours :
> il faut régulariser, soit par une pénalité, soit en arrêtant l'optimisation tôt (section 4).